In [3]:
import os
import cv2
import xml.etree.ElementTree as ET

# ------------------------------------------------------------
# INPUT FOLDERS
# ------------------------------------------------------------
IMAGE_FOLDER = r"archive(1)/images"
XML_FOLDER = r"archive(1)/annotations"

# ------------------------------------------------------------
# OUTPUT FOLDERS
# ------------------------------------------------------------
WITH_MASK_FOLDER = r"facemask-dataset/with_mask"
WITHOUT_MASK_FOLDER = r"facemask-dataset/without_mask"

os.makedirs(WITH_MASK_FOLDER, exist_ok=True)
os.makedirs(WITHOUT_MASK_FOLDER, exist_ok=True)

# ------------------------------------------------------------
# READ EACH XML FILE
# ------------------------------------------------------------
for xml_file in os.listdir(XML_FOLDER):

    if not xml_file.endswith(".xml"):
        continue

    xml_path = os.path.join(XML_FOLDER, xml_file)

    tree = ET.parse(xml_path)
    root = tree.getroot()

    # Read image name from XML
    filename = root.find("filename").text
    image_path = os.path.join(IMAGE_FOLDER, filename)

    image = cv2.imread(image_path)

    if image is None:
        print("Image not found:", image_path)
        continue

    count = 0

    # Read every detected face in the image
    for obj in root.findall("object"):

        label = obj.find("name").text.strip().lower()

        bbox = obj.find("bndbox")

        xmin = int(bbox.find("xmin").text)
        ymin = int(bbox.find("ymin").text)
        xmax = int(bbox.find("xmax").text)
        ymax = int(bbox.find("ymax").text)

        # Crop the face
        face = image[ymin:ymax, xmin:xmax]

        if face.size == 0:
            continue

        # Name of cropped face file
        save_name = f"{os.path.splitext(filename)[0]}_{count}.png"

        # Save in proper folder
        if label == "without_mask":
            save_path = os.path.join(WITHOUT_MASK_FOLDER, save_name)
        else:
            # with_mask and mask_weared_incorrect both go here
            save_path = os.path.join(WITH_MASK_FOLDER, save_name)

        cv2.imwrite(save_path, face)
        count += 1

print("Done! Cropped faces are saved in:")
print(WITH_MASK_FOLDER)
print(WITHOUT_MASK_FOLDER)

Done! Cropped faces are saved in:
facemask-dataset/with_mask
facemask-dataset/without_mask


In [7]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# ------------------------------------------------------------
# DATASET FOLDERS
# ------------------------------------------------------------
WITH_MASK_FOLDER = r"facemask-dataset/with_mask"
WITHOUT_MASK_FOLDER = r"facemask-dataset/without_mask"

# ------------------------------------------------------------
# LOAD IMAGES
# ------------------------------------------------------------
X = []
y = []

# with_mask -> label = 1
for file in os.listdir(WITH_MASK_FOLDER):

    path = os.path.join(WITH_MASK_FOLDER, file)

    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)

    if img is None:
        continue

    # Convert to 64x64
    img = cv2.resize(img, (64, 64))

    # Flatten into 1-D feature vector
    img = img.flatten()

    X.append(img)
    y.append(1)

# without_mask -> label = 0
for file in os.listdir(WITHOUT_MASK_FOLDER):

    path = os.path.join(WITHOUT_MASK_FOLDER, file)

    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)

    if img is None:
        continue

    img = cv2.resize(img, (64, 64))
    img = img.flatten()

    X.append(img)
    y.append(0)

# Convert into NumPy arrays
X = np.array(X)
y = np.array(y)

# Scale pixel values
X = X / 255.0

# Reduce 4096 features to 100 important features

pca = PCA(n_components=200)
X = pca.fit_transform(X)

print("Total Images:", len(X))
print("With Mask:", np.sum(y == 1))
print("Without Mask:", np.sum(y == 0))

# ------------------------------------------------------------
# TRAIN-TEST SPLIT (70:30)
# ------------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

print("Training Images:", len(X_train))
print("Testing Images:", len(X_test))

# ------------------------------------------------------------
# TRAIN LOGISTIC REGRESSION MODEL
# ------------------------------------------------------------
model = LogisticRegression(max_iter=5000)

model.fit(X_train, y_train)

# ------------------------------------------------------------
# PREDICT
# ------------------------------------------------------------
y_pred = model.predict(X_test)

# ------------------------------------------------------------
# EVALUATION
# ------------------------------------------------------------
print("\nAccuracy:")
print(accuracy_score(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=["Without Mask", "With Mask"]
))

Total Images: 7905
With Mask: 5270
Without Mask: 2635
Training Images: 5533
Testing Images: 2372

Accuracy:
0.75

Confusion Matrix:
[[ 392  399]
 [ 194 1387]]

Classification Report:
              precision    recall  f1-score   support

Without Mask       0.67      0.50      0.57       791
   With Mask       0.78      0.88      0.82      1581

    accuracy                           0.75      2372
   macro avg       0.72      0.69      0.70      2372
weighted avg       0.74      0.75      0.74      2372



In [8]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split

# ------------------------------------------------------------
# DATASET PATH
# facemask-dataset/
# ├── with_mask/
# └── without_mask/
# ------------------------------------------------------------
DATASET_PATH = "facemask-dataset"

# ------------------------------------------------------------
# IMAGE TRANSFORMATIONS
# ------------------------------------------------------------
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# ------------------------------------------------------------
# LOAD DATASET
# ------------------------------------------------------------
dataset = datasets.ImageFolder(DATASET_PATH, transform=transform)

# 70:30 split
train_size = int(0.7 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print("Classes:", dataset.classes)
print("Train Images:", len(train_dataset))
print("Test Images:", len(test_dataset))

# ------------------------------------------------------------
# LOAD PRETRAINED MOBILENETV2
# ------------------------------------------------------------
model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)

# Replace final layer
model.classifier[1] = nn.Linear(model.last_channel, 2)

# ------------------------------------------------------------
# DEVICE
# ------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# ------------------------------------------------------------
# LOSS AND OPTIMIZER
# ------------------------------------------------------------
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

# ------------------------------------------------------------
# TRAIN MODEL
# ------------------------------------------------------------
epochs = 5

for epoch in range(epochs):

    model.train()
    running_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total

    print(f"Epoch {epoch+1}/{epochs}")
    print(f"Loss: {running_loss:.4f}")
    print(f"Training Accuracy: {accuracy:.2f}%")

# ------------------------------------------------------------
# TEST MODEL
# ------------------------------------------------------------
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

test_accuracy = 100 * correct / total

print(f"\nTest Accuracy: {test_accuracy:.2f}%")

# ------------------------------------------------------------
# SAVE MODEL
# ------------------------------------------------------------
torch.save(model.state_dict(), "mask_detector_mobilenetv2.pth")

print("Model saved as mask_detector_mobilenetv2.pth")

Classes: ['with_mask', 'without_mask']
Train Images: 5533
Test Images: 2372
Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to C:\Users\SHUBHANKAR/.cache\torch\hub\checkpoints\mobilenet_v2-7ebf99e0.pth


100.0%
c:\Users\SHUBHANKAR\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\PIL\Image.py:1137: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Epoch 1/5
Loss: 33.1744
Training Accuracy: 93.33%
Epoch 2/5
Loss: 7.2331
Training Accuracy: 98.63%
Epoch 3/5
Loss: 5.4391
Training Accuracy: 99.06%
Epoch 4/5
Loss: 3.6722
Training Accuracy: 99.37%
Epoch 5/5
Loss: 2.6972
Training Accuracy: 99.46%

Test Accuracy: 98.74%
Model saved as mask_detector_mobilenetv2.pth
